In [ ]:
"""
TinyLlama 1.1B QLoRA Fine-tuning on EmpatheticDialogues
Optimized for edge deployment (GGUF/ONNX quantization)
Run: python train_tinyllama.py
"""

import os
import torch
import pandas as pd
import logging
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    TaskType,
)
from trl import SFTTrainer
from transformers import DataCollatorForLanguageModeling

In [ ]:
# ─────────────────────────────────────────────
# CONFIG — change these as needed
# ─────────────────────────────────────────────
CONFIG = {
    "base_model":       "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "output_dir":       "./tinyllama-qlora-emotional-v3",
    "final_model_dir":  "./tinyllama-emotional-final",
    "max_seq_length":   512,
    "learning_rate":    5e-5,
    "num_epochs":       2,
    "batch_size":       4,
    "grad_accum":       8,           # effective batch = 32
    "eval_steps":       100,
    "save_steps":       100,         # must match eval_steps
    "lora_r":           16,
    "lora_alpha":       32,
    "lora_dropout":     0.1,
    "patience":         4,           # early stopping patience
    "log_file":         "training.log",
}

# ─────────────────────────────────────────────
# LOGGING SETUP
# ─────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(CONFIG["log_file"]),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

In [ ]:
# ─────────────────────────────────────────────
# STEP 1 — GPU CHECK
# ─────────────────────────────────────────────
def check_gpu():
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU found. This script requires a CUDA GPU.")
    log.info(f"GPU        : {torch.cuda.get_device_name(0)}")
    log.info(f"VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    log.info(f"PyTorch    : {torch.__version__}")

In [ ]:
# ─────────────────────────────────────────────
# STEP 2 — DATASET LOADING & CLEANING
# ─────────────────────────────────────────────
def build_conversation_pairs(split):
    """
    Build correct user/bot pairs from EmpatheticDialogues
    using conv_id to avoid first-person confusion.
    Even utterance_idx = user, Odd = assistant.
    """
    df = pd.DataFrame(split)

    # Fix _comma_ artifact in dataset
    df["utterance"] = df["utterance"].str.replace("_comma_", ",", regex=False)
    df["prompt"]    = df["prompt"].str.replace("_comma_", ",", regex=False)

    pairs = []
    for conv_id, group in df.groupby("conv_id"):
        group   = group.sort_values("utterance_idx").reset_index(drop=True)
        emotion = group.iloc[0]["context"]

        for i in range(0, len(group) - 1, 2):
            user_turn = group.iloc[i]["utterance"].strip()
            bot_turn  = group.iloc[i + 1]["utterance"].strip()

            if not user_turn or not bot_turn:
                continue

            pairs.append({
                "user":    user_turn,
                "bot":     bot_turn,
                "emotion": emotion,
            })

    return pairs

In [ ]:
def load_data():
    log.info("Loading EmpatheticDialogues dataset...")
    dataset = load_dataset("facebook/empathetic_dialogues")

    train_pairs = build_conversation_pairs(dataset["train"])
    val_pairs   = build_conversation_pairs(dataset["validation"])

    log.info(f"Train pairs : {len(train_pairs)}")
    log.info(f"Val pairs   : {len(val_pairs)}")
    log.info(f"Sample user : {train_pairs[0]['user']}")
    log.info(f"Sample bot  : {train_pairs[0]['bot']}")

    return train_pairs, val_pairs

In [ ]:
# ─────────────────────────────────────────────
# STEP 3 — PROMPT FORMATTING
# ─────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are an empathetic AI assistant. "
    "Listen carefully and respond with warmth and emotional intelligence. "
    "Never repeat what the user said. Never speak as the user. "
    "Never share personal experiences. Focus entirely on the user's feelings."
)

def format_pairs(pairs, tokenizer):
    formatted = []
    for p in pairs:
        prompt = (
            f"<|system|>\n{SYSTEM_PROMPT}\n"
            f"<|user|>\n[Emotion: {p['emotion']}] {p['user']}\n"
            f"<|assistant|>\n{p['bot']}{tokenizer.eos_token}"  # EOS stops rambling
        )
        formatted.append({"text": prompt})
    return formatted


In [ ]:
# ─────────────────────────────────────────────
# STEP 4 — MODEL LOADING (4-bit QLoRA)
# ─────────────────────────────────────────────
def load_model_and_tokenizer():
    log.info(f"Loading base model: {CONFIG['base_model']}")

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = "right"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        CONFIG["base_model"],
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )

    model.config.use_cache       = False
    model.config.pretraining_tp  = 1

    log.info(f"Model loaded. Parameters: {model.num_parameters():,}")
    return model, tokenizer



In [ ]:
# ─────────────────────────────────────────────
# STEP 5 — LORA ADAPTER CONFIG
# ─────────────────────────────────────────────
def apply_lora(model):
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        target_modules=[
            "q_proj", "k_proj",
            "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        lora_dropout=CONFIG["lora_dropout"],
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


In [ ]:
# ─────────────────────────────────────────────
# STEP 6 — CHECKPOINT BACKUP CALLBACK
# ─────────────────────────────────────────────
class CheckpointBackupCallback(TrainerCallback):
    """
    Backs up every checkpoint to a separate safe folder.
    Change backup_dir to a mounted Drive path if on Colab:
    backup_dir = "/content/drive/MyDrive/tinyllama_training"
    """
    def __init__(self, backup_dir="./checkpoints-backup"):
        self.backup_dir = backup_dir
        os.makedirs(backup_dir, exist_ok=True)

    def on_save(self, args, state, control, **kwargs):
        import shutil
        src = f"{args.output_dir}/checkpoint-{state.global_step}"
        dst = f"{self.backup_dir}/checkpoint-{state.global_step}"
        if os.path.exists(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            log.info(f"Backed up checkpoint-{state.global_step} → {dst} ✅")


In [ ]:
# ─────────────────────────────────────────────
# STEP 7 — TRAINING
# ─────────────────────────────────────────────
def train(model, tokenizer, train_dataset, val_dataset):
    training_args = TrainingArguments(
        output_dir=CONFIG["output_dir"],
        num_train_epochs=CONFIG["num_epochs"],
        per_device_train_batch_size=CONFIG["batch_size"],
        per_device_eval_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["grad_accum"],
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        learning_rate=CONFIG["learning_rate"],
        weight_decay=0.001,
        fp16=True,
        bf16=False,
        max_grad_norm=0.3,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        evaluation_strategy="steps",
        eval_steps=CONFIG["eval_steps"],
        save_strategy="steps",
        save_steps=CONFIG["save_steps"],
        save_total_limit=3,
        logging_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        group_by_length=True,
    )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        dataset_text_field="text",
        max_seq_length=CONFIG["max_seq_length"],
        packing=False,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=CONFIG["patience"],
                early_stopping_threshold=0.005,
            ),
            CheckpointBackupCallback(
                backup_dir="./checkpoints-backup"
                # For Colab use:
                # backup_dir="/content/drive/MyDrive/tinyllama_training"
            ),
        ]
    )

    log.info("Starting training...")
    trainer.train()
    log.info("Training complete ✅")
    return trainer


In [ ]:
# ─────────────────────────────────────────────
# STEP 8 — SAVE FINAL MODEL
# ─────────────────────────────────────────────
def save_final_model(model, tokenizer):
    log.info(f"Saving final model to {CONFIG['final_model_dir']}")
    model.save_pretrained(CONFIG["final_model_dir"])
    tokenizer.save_pretrained(CONFIG["final_model_dir"])
    log.info("Final model saved ✅")


In [ ]:
# ─────────────────────────────────────────────
# STEP 9 — MERGE LORA FOR EDGE EXPORT
# ─────────────────────────────────────────────
def merge_and_export(tokenizer):
    log.info("Merging LoRA adapters into base model for edge export...")

    base_model = AutoModelForCausalLM.from_pretrained(
        CONFIG["base_model"],
        torch_dtype=torch.float16,
        device_map="cpu",            # CPU merge = safer, less VRAM
    )

    merged_model = PeftModel.from_pretrained(
        base_model,
        CONFIG["final_model_dir"],
        local_files_only=True,
    )
    merged_model = merged_model.merge_and_unload()

    merged_path = "./tinyllama-emotional-merged"
    merged_model.save_pretrained(merged_path, safe_serialization=True)
    tokenizer.save_pretrained(merged_path)
    log.info(f"Merged model saved to {merged_path} ✅")
    log.info("Ready for GGUF/ONNX quantization")
    return merged_path


In [ ]:
# ─────────────────────────────────────────────
# STEP 10 — INFERENCE TEST
# ─────────────────────────────────────────────
def chat(model, tokenizer, user_input, emotion=None,
         max_new_tokens=100, temperature=0.7):

    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n"
        f"{'[Emotion: ' + emotion + '] ' if emotion else ''}"
        f"{user_input}\n"
        f"<|assistant|>\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    # Hard stop at role leakage
    for stop_token in ["<|user|>", "<|system|>", "<|assistant|>"]:
        if stop_token in response:
            response = response.split(stop_token)[0]

    return response.strip()



In [ ]:
def run_inference_tests(model, tokenizer):
    log.info("\n" + "="*60)
    log.info("INFERENCE TESTS")
    log.info("="*60)

    test_cases = [
        ("I got promoted today, I can't believe it!",      "joy"),
        ("I've been feeling really lonely lately.",        "sadness"),
        ("My boss blamed me for something I didn't do!",   "anger"),
        ("I have a big exam tomorrow and I'm terrified.",  "fear"),
        ("I just bumped into my ex at the mall.",          "surprise"),
        ("I hate how people litter everywhere.",           "disgust"),
        ("What did you do over the weekend?",               None),
    ]

    for user_input, emotion in test_cases:
        response = chat(model, tokenizer, user_input, emotion)
        log.info(f"\n🧑 [{emotion or 'no tag'}] {user_input}")
        log.info(f"🤖 {response}")
        log.info("-" * 60)



In [ ]:
# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    log.info("=" * 60)
    log.info("TinyLlama QLoRA Emotional Fine-tuning")
    log.info("=" * 60)

    # 1. GPU check
    check_gpu()

    # 2. Load dataset
    train_pairs, val_pairs = load_data()

    # 3. Load model + tokenizer
    model, tokenizer = load_model_and_tokenizer()

    # 4. Format datasets
    train_dataset = Dataset.from_list(format_pairs(train_pairs, tokenizer))
    val_dataset   = Dataset.from_list(format_pairs(val_pairs,   tokenizer))
    log.info(f"Formatted — Train: {len(train_dataset)} | Val: {len(val_dataset)}")

    # 5. Apply LoRA
    model = apply_lora(model)

    # 6. Train
    trainer = train(model, tokenizer, train_dataset, val_dataset)

    # 7. Save adapter
    save_final_model(trainer.model, tokenizer)

    # 8. Run inference tests
    trainer.model.eval()
    run_inference_tests(trainer.model, tokenizer)

    # 9. Merge for edge export
    merge_and_export(tokenizer)

    log.info("\n✅ All done! Model ready for GGUF quantization.")
    log.info(f"Merged model: ./tinyllama-emotional-merged")
    log.info(f"Logs saved:   {CONFIG['log_file']}")


if __name__ == "__main__":
    main()